<a href="https://colab.research.google.com/github/ndremferraz/DeepVINS/blob/main/DeepVINS_trainning_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepVINS Trainning V2

This document contains the entirety of the code used to train DeepVINS, I understand that this may be a little bit unorthodox. Nevertheless, having everything on a single notebook like this help me streamline development significantly.  

In [ ]:
import torch
import torch.nn as nn
import gc
import random
from pathlib import Path

## Dataset Loading and batching

The EurocMav_Batcher class was the solution that I found to deal with this massive datasets. It creates a validation dataset and allow us to get batches from the different sequences in a Round Robin fashion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/MH_01_easy_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/MH_02_easy_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/MH_03_medium_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/MH_04_difficult_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/MH_05_difficult_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/V1_01_easy_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/V1_02_medium_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/V1_03_difficult_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/V2_01_easy_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/V2_02_medium_dataset.pt" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DeepVINS Datasets/V2_03_difficult_dataset.pt" "/content"

Mounted at /content/drive


In [ ]:
class EurocMav_Batcher:
  def __init__(self, dataset_files, batch_size = 128):
    self.dataset_files = dataset_files
    self.batch_size = batch_size
    self.batch_number = 0
    self.dataset_idx = 0

    self.loaded_dataset = None
    self.valid_batch_idxs = [None for _ in self.dataset_files]
    self.valid_batch = self.create_valid_batch()

  def reset(self):
    self.batch_number = 0
    self.dataset_idx = 0

  def create_valid_batch(self):
    img_batches = []
    imu_batches = []
    target_batches = []

    for dataset_idx in range(len(self.dataset_files)):
      self._load_dataset(dataset_idx)

      images = self.loaded_dataset["images"]
      imus = self.loaded_dataset["imus"]
      gts = self.loaded_dataset["gts"]

      num_examples = images.shape[0]

      num_batches = (num_examples - 1) // self.batch_size
      start = random.randint(0, max(0, num_batches - 1))
      self.valid_batch_idxs[dataset_idx] = start

      start = start * self.batch_size + 1
      end = start + self.batch_size

      img_batches.append(images[start: end])
      imu_batches.append(imus[start: end])
      target_batches.append(gts[start - 1: end])

    self.loaded_dataset = None
    gc.collect()

    return {
        "img_batch": torch.cat(img_batches, dim=0),
        "imu_batch": torch.cat(imu_batches, dim=0),
        "target_batch": torch.cat(target_batches, dim=0),
    }

  def get_valid_batch(self):
    return self.valid_batch

  def _load_dataset(self, dataset_idx):
    self.loaded_dataset = None
    gc.collect()
    self.loaded_dataset = torch.load(f"/content/{self.dataset_files[dataset_idx]}")

  def _advance_round_robin(self):
    if self.dataset_idx >= len(self.dataset_files) - 1:
        self.dataset_idx = 0
        self.batch_number += 1
    else:
        self.dataset_idx += 1

  def is_valid_batch(self, dataset_idx, batch_number):
    batch_valid = self.valid_batch_idxs[dataset_idx]
    return batch_number == batch_valid

  def get_batch(self):
    attempts = 0

    while attempts < len(self.dataset_files):
        dataset_idx = self.dataset_idx
        batch_number = self.batch_number

        self._load_dataset(dataset_idx)

        images = self.loaded_dataset["images"]
        imu = self.loaded_dataset["imus"]
        target = self.loaded_dataset["gts"]

        num_examples = images.shape[0]

        start = batch_number * self.batch_size + 1
        end = start + self.batch_size

        self._advance_round_robin()

        if self.is_valid_batch(dataset_idx, batch_number):
            attempts += 1
            continue

        if end > num_examples:
            attempts += 1
            continue

        return {
            "img_batch": images[start:end],
            "imu_batch": imu[start:end],
            "target_batch": target[start - 1:end],
        }

    return None


In [ ]:
DATASET_FILES = [
    "MH_01_easy_dataset.pt",
    "MH_02_easy_dataset.pt",
    "MH_03_medium_dataset.pt",
    "MH_04_difficult_dataset.pt",
    "MH_05_difficult_dataset.pt",
    "V1_01_easy_dataset.pt",
    "V1_02_medium_dataset.pt",
    "V1_03_difficult_dataset.pt",
    "V2_01_easy_dataset.pt",
    "V2_02_medium_dataset.pt",
    "V2_03_difficult_dataset.pt",
]

batcher = EurocMav_Batcher(DATASET_FILES, batch_size=64)
valid_batch = batcher.get_valid_batch()

batch01 = batcher.get_batch()
batch02 = batcher.get_batch()
batch03 = batcher.get_batch()
batch04 = batcher.get_batch()

print(valid_batch["img_batch"].shape)
print(batch01["img_batch"].shape)
print(batch02["img_batch"].shape)
print(batch03["img_batch"].shape)
print(batch04["img_batch"].shape)
print(batch01["imu_batch"].shape)

torch.Size([704, 2, 480, 752])
torch.Size([64, 2, 480, 752])
torch.Size([64, 2, 480, 752])
torch.Size([64, 2, 480, 752])
torch.Size([64, 2, 480, 752])
torch.Size([64, 10, 6])


In [ ]:
print(batch01["target_batch"].shape)

torch.Size([65, 7])


## Implementation of Inertial and Visual encoders

In [ ]:
class ImageEnconder(nn.Module):

  def __init__(self,  img_channels: int = 2,
                      l1_filters: int = 8, l2_filters: int = 16, l3_filters: int = 32,
                      l4_dims: int = 64, l5_dims: int = 128, l6_dims: int = 512,
                      fc_dims: int = 512):

    super().__init__()

    l1_size = img_channels * l1_filters
    l2_size = l1_size * l2_filters
    l3_size = l2_size * l3_filters

    # Three depthwise stages let each image develop independent low-level features
    # before the later layers start mixing the pair together.
    self.l1 = nn.Sequential(
            nn.Conv2d(
                in_channels=img_channels,
                out_channels=l1_size,
                kernel_size=5,
                stride=2,
                groups=img_channels
            ),
            nn.BatchNorm2d(l1_size),
            nn.LeakyReLU(0.01)
    )
    self.l2 = nn.Sequential(
            nn.Conv2d(
                in_channels=l1_size,
                out_channels=l2_size,
                kernel_size=5,
                stride=2,
                groups=l1_size
            ),
            nn.BatchNorm2d(l2_size),
            nn.LeakyReLU(0.01)
    )
    self.avg_pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
    self.l3 = nn.Sequential(
            nn.Conv2d(
                in_channels=l2_size,
                out_channels=l3_size,
                kernel_size=3,
                stride=2,
                groups=l2_size
            ),
            nn.BatchNorm2d(l3_size),
            nn.LeakyReLU(0.01)
    )

    #Regular 2D convolutions followed by Fully connected layers
    self.l4 = nn.Sequential(
            nn.Conv2d(
                in_channels=l3_size,
                out_channels=l4_dims,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(l4_dims),
            nn.LeakyReLU(0.01)
    )
    self.avg_pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
    self.l5 = nn.Sequential(
          nn.Conv2d(
              in_channels=l4_dims,
              out_channels=l5_dims,
              kernel_size=3,
              stride=2,
              padding=1
          ),
          nn.BatchNorm2d(l5_dims),
          nn.LeakyReLU(0.01)
    )
    self.l6 = nn.Sequential(
        nn.Conv2d(
            in_channels=l5_dims,
            out_channels=l6_dims,
            kernel_size=3,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(l6_dims),
        nn.LeakyReLU(0.01)
    )
    self.spatial_pool = nn.AdaptiveAvgPool2d((1, 1))

    self.fc = nn.Sequential(
        nn.Linear(l6_dims, fc_dims),
        nn.LayerNorm(fc_dims),
        nn.LeakyReLU(0.01)
    )

  def forward(self, x: torch.Tensor):
    x = self.l1(x)
    x = self.l2(x)
    x = self.avg_pool1(x)
    x = self.l3(x)
    x = self.l4(x)
    x = self.avg_pool2(x)
    x = self.l5(x)
    x = self.l6(x)
    x = self.spatial_pool(x)
    x = torch.flatten(x, start_dim=1)
    x = self.fc(x)
    return x

In [ ]:
class InertialEncoder(nn.Module):

  def __init__(
      self,
      imu_steps: int = 10,
      l1_filters: int = 8, l2_filters: int = 16,
      l3_dims: int = 64, l4_dims: int = 256,
      fc_dims: int = 512
  ):

    super().__init__()

    l1_size = imu_steps * l1_filters
    l2_size = l1_size * l2_filters

    # Depthwise convolution is applied with the intent to filter each imu channel individually
    self.l1 = nn.Sequential(
        nn.Conv1d(
            in_channels=imu_steps,
            out_channels=l1_size,
            kernel_size=3,
            padding=1,
            groups=imu_steps
        ),
        nn.BatchNorm1d(l1_size),
        nn.LeakyReLU(0.01)
    )
    self.l2 = nn.Sequential(
        nn.Conv1d(
            in_channels=l1_size,
            out_channels=l2_size,
            kernel_size=3,
            padding=1,
            groups=l1_size
        ),
        nn.BatchNorm1d(l2_size),
        nn.LeakyReLU(0.01)
    )

    # Regular 1D convolution followed by Fully connected layers
    self.l3 = nn.Sequential(
        nn.Conv1d(
            in_channels=l2_size,
            out_channels=l3_dims,
            kernel_size=3,
            padding=1
        ),
        nn.BatchNorm1d(l3_dims),
        nn.LeakyReLU(0.01)
    )
    self.l4 = nn.Sequential(
        nn.Conv1d(
            in_channels=l3_dims,
            out_channels=l4_dims,
            kernel_size=3,
            padding=1
        ),
        nn.BatchNorm1d(l4_dims),
        nn.LeakyReLU(0.01)
    )

    self.spatial_pool = nn.AdaptiveAvgPool1d(1)

    self.fc = nn.Sequential(
        nn.Linear(l4_dims, fc_dims),
        nn.LayerNorm(fc_dims),
        nn.LeakyReLU(0.01)
    )

  def forward(self, x: torch.Tensor):
      x = self.l1(x)
      x = self.l2(x)
      x = self.l3(x)
      x = self.l4(x)
      x = self.spatial_pool(x)
      x = torch.flatten(x, start_dim=1)
      x = self.fc(x)
      return x


## Implementation of Visual Inertial Fusion Transformer

In [ ]:
class AttentionBlock(nn.Module):
  def __init__(self, embed_dim=1024, num_heads=4, dropout=0.1):
      super().__init__()

      self.norm1 = nn.LayerNorm(embed_dim)
      self.dropout = nn.Dropout(dropout)

      self.attn = nn.MultiheadAttention(
          embed_dim=embed_dim,
          num_heads=num_heads,
          dropout=dropout,
          batch_first=True
      )

      self.norm2 = nn.LayerNorm(embed_dim)
      self.l2 = nn.Sequential(
          nn.Linear(embed_dim, 4 * embed_dim),
          nn.GELU(),
          nn.Dropout(dropout),
          nn.Linear(4 * embed_dim, embed_dim),
          nn.Dropout(dropout)
      )

  def forward(self, x):
      t = x.size(1)
      causal_mask = torch.triu(
            torch.ones(t, t, device=x.device, dtype=torch.bool),
            diagonal=1
      )

      attn_input = self.norm1(x)
      attn_output, _ = self.attn(
          query=attn_input,
          key=attn_input,
          value=attn_input,
          attn_mask=causal_mask,
          need_weights=False
      )
      x = x + self.dropout(attn_output)
      mlp_input = self.norm2(x)
      mlp_output = self.l2(mlp_input)
      x = x + self.dropout(mlp_output)

      return x

In [ ]:
class CausalFusionModel(nn.Module):

  def __init__(
      self,
      imu_steps: int = 10,
      img_channels: int = 2,
      attn_blocks: int = 4,
      embed_dim: int = 1024,
      num_heads: int = 4,
      context_length: int = 64,
      output_dim: int = 6,
      dropout: float = 0.1
  ):

    super().__init__()

    self.imu_encoder = InertialEncoder(imu_steps=imu_steps,fc_dims=embed_dim // 2)
    self.img_encoder = ImageEnconder(img_channels=img_channels,fc_dims=embed_dim // 2)

    self.pos_emb = nn.Embedding(context_length, embed_dim)

    self.attn_blocks = nn.ModuleList(
        [AttentionBlock(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout)
        for _ in range(attn_blocks)]
    )

    self.norm = nn.LayerNorm(embed_dim)
    self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, output_dim)
    )
    self._init_weights()

  def _init_weights(self):
      for module in self.modules():
          if isinstance(module, (nn.Conv1d, nn.Conv2d)):
              nn.init.kaiming_normal_(module.weight, nonlinearity="leaky_relu")
              if module.bias is not None:
                  nn.init.zeros_(module.bias)
          elif isinstance(module, nn.Linear):
              nn.init.xavier_uniform_(module.weight)
              if module.bias is not None:
                  nn.init.zeros_(module.bias)
          elif isinstance(module, nn.MultiheadAttention):
              nn.init.xavier_uniform_(module.in_proj_weight)
              if module.in_proj_bias is not None:
                  nn.init.zeros_(module.in_proj_bias)
              nn.init.xavier_uniform_(module.out_proj.weight)
              if module.out_proj.bias is not None:
                  nn.init.zeros_(module.out_proj.bias)
          elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.LayerNorm)):
              if module.weight is not None:
                  nn.init.ones_(module.weight)
              if module.bias is not None:
                  nn.init.zeros_(module.bias)


  def forward(self, img_batch: torch.Tensor, imu_batch: torch.Tensor):

      img_out = self.img_encoder(img_batch)
      imu_out = self.imu_encoder(imu_batch)

      fused = torch.cat((imu_out, img_out), dim=1)
      tokens = fused.unsqueeze(0)

      T = tokens.shape[1]

      pos = torch.arange(T, device=tokens.device)
      pos_emb = self.pos_emb(pos)

      tokens = tokens + pos_emb

      for attn_block in self.attn_blocks:
          tokens = attn_block(tokens)

      tokens = self.norm(tokens)
      tokens = self.head(tokens)

      return tokens

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CausalFusionModel()
model = model.to(device)

img = batch01["img_batch"]
imu = batch01["imu_batch"]

img = img.to(device)
imu = imu.to(device)

with torch.no_grad():
    out = model(img, imu)

print(out.shape)

torch.Size([1, 64, 6])


## Loss function implementation


The helper functions related to geometrical conversions were all generated by were generated by GPT 5.4

In [ ]:
EPS = 1e-6

def normalize_quaternion(quaternion: torch.Tensor) -> torch.Tensor:
    return quaternion / quaternion.norm(dim=-1, keepdim=True).clamp_min(1e-8)

def quaternion_to_euler(quaternion: torch.Tensor) -> torch.Tensor:
    quaternion = normalize_quaternion(quaternion)
    qw, qx, qy, qz = quaternion.unbind(dim=-1)

    sinr_cosp = 2.0 * (qw * qx + qy * qz)
    cosr_cosp = 1.0 - 2.0 * (qx * qx + qy * qy)
    roll = torch.atan2(sinr_cosp, cosr_cosp)

    sinp = 2.0 * (qw * qy - qz * qx)
    pitch = torch.asin(sinp.clamp(-1.0 + EPS, 1.0 - EPS))

    siny_cosp = 2.0 * (qw * qz + qx * qy)
    cosy_cosp = 1.0 - 2.0 * (qy * qy + qz * qz)
    yaw = torch.atan2(siny_cosp, cosy_cosp)

    return torch.stack((roll, pitch, yaw), dim=-1)

def euler_to_rotation_matrix(euler: torch.Tensor) -> torch.Tensor:
    roll, pitch, yaw = euler.unbind(dim=-1)

    cr = torch.cos(roll)
    sr = torch.sin(roll)
    cp = torch.cos(pitch)
    sp = torch.sin(pitch)
    cy = torch.cos(yaw)
    sy = torch.sin(yaw)

    r00 = cy * cp
    r01 = cy * sp * sr - sy * cr
    r02 = cy * sp * cr + sy * sr

    r10 = sy * cp
    r11 = sy * sp * sr + cy * cr
    r12 = sy * sp * cr - cy * sr

    r20 = -sp
    r21 = cp * sr
    r22 = cp * cr

    row0 = torch.stack((r00, r01, r02), dim=-1)
    row1 = torch.stack((r10, r11, r12), dim=-1)
    row2 = torch.stack((r20, r21, r22), dim=-1)
    return torch.stack((row0, row1, row2), dim=-2)

def rotation_matrix_to_euler(rotation: torch.Tensor) -> torch.Tensor:
    sy = torch.sqrt((rotation[..., 0, 0] ** 2 + rotation[..., 1, 0] ** 2).clamp_min(EPS))
    singular = sy < 1e-6

    roll = torch.atan2(rotation[..., 2, 1], rotation[..., 2, 2])
    pitch = torch.atan2(-rotation[..., 2, 0], sy)
    yaw = torch.atan2(rotation[..., 1, 0], rotation[..., 0, 0])

    singular_roll = torch.atan2(-rotation[..., 1, 2], rotation[..., 1, 1])
    singular_pitch = torch.atan2(-rotation[..., 2, 0], sy.clamp_min(EPS))
    singular_yaw = torch.zeros_like(yaw)

    roll = torch.where(singular, singular_roll, roll)
    pitch = torch.where(singular, singular_pitch, pitch)
    yaw = torch.where(singular, singular_yaw, yaw)

    return torch.stack((roll, pitch, yaw), dim=-1)

def euler_to_quaternion(euler: torch.Tensor) -> torch.Tensor:
    roll, pitch, yaw = euler.unbind(dim=-1)

    half_roll = roll * 0.5
    half_pitch = pitch * 0.5
    half_yaw = yaw * 0.5

    cr = torch.cos(half_roll)
    sr = torch.sin(half_roll)
    cp = torch.cos(half_pitch)
    sp = torch.sin(half_pitch)
    cy = torch.cos(half_yaw)
    sy = torch.sin(half_yaw)

    qw = cr * cp * cy + sr * sp * sy
    qx = sr * cp * cy - cr * sp * sy
    qy = cr * sp * cy + sr * cp * sy
    qz = cr * cp * sy - sr * sp * cy

    quaternion = torch.stack((qw, qx, qy, qz), dim=-1)
    return normalize_quaternion(quaternion)


def split_prediction(prediction: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:

  translation = prediction[..., :3]
  euler = prediction[..., 3:]
  quaternion = euler_to_quaternion(euler)
  return translation, quaternion

def split_pose(gt_pose: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:

  translation = gt_pose[..., :3]
  quaternion = gt_pose[..., 3:]

  return translation, quaternion

def prediction_to_transform(prediction: torch.Tensor) -> torch.Tensor:

    translation, quaternion = split_prediction(prediction)
    euler = quaternion_to_euler(quaternion)
    rotation = euler_to_rotation_matrix(euler)

    batch, steps = translation.shape[:2]
    transform = torch.eye(4, device=prediction.device, dtype=prediction.dtype).repeat(batch, steps, 1, 1)
    transform[..., :3, :3] = rotation
    transform[..., :3, 3] = translation
    return transform

def pose_to_transform(pose: torch.Tensor) -> torch.Tensor:
    translation, quaternion = split_pose(pose)
    euler = quaternion_to_euler(quaternion)
    rotation = euler_to_rotation_matrix(euler)

    batch, steps = translation.shape[:2]
    transform = torch.eye(4, device=pose.device, dtype=pose.dtype).repeat(batch, steps, 1, 1)
    transform[..., :3, :3] = rotation
    transform[..., :3, 3] = translation
    return transform

def invert_transform(transform: torch.Tensor) -> torch.Tensor:
    rotation = transform[..., :3, :3]
    translation = transform[..., :3, 3]

    rotation_inv = rotation.transpose(-1, -2)
    translation_inv = -(rotation_inv @ translation.unsqueeze(-1)).squeeze(-1)

    transform_inv = torch.eye(4, device=transform.device, dtype=transform.dtype).repeat(*transform.shape[:-2], 1, 1)
    transform_inv[..., :3, :3] = rotation_inv
    transform_inv[..., :3, 3] = translation_inv
    return transform_inv

def transform_to_components(transform: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    translation = transform[..., :3, 3]
    euler = rotation_matrix_to_euler(transform[..., :3, :3])
    return translation, euler

def compose_sequence_transforms(transform: torch.Tensor) -> torch.Tensor:
    transform = transform.clone()
    cumulative = []
    running = torch.eye(4, device=transform.device, dtype=transform.dtype).repeat(transform.size(0), 1, 1)

    for step in range(transform.size(1)):
        running = running @ transform[:, step]
        cumulative.append(running)

    return torch.stack(cumulative, dim=1)

def rmse(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean((prediction - target) ** 2) + 1e-8)

class PoseSequenceLoss(nn.Module):

  def __init__(
      self,
      frame_weight: float = 0.90,
      sequence_weight: float = 0.10,
      translation_weight: float = 0.5,
      rotation_weight: float = 0.5
  ):

    super().__init__()

    self.frame_weight = frame_weight
    self.sequence_weight = sequence_weight
    self.translation_weight = translation_weight
    self.rotation_weight = rotation_weight

  def forward(
    self,
    prediction: torch.Tensor,
    target: torch.Tensor
  ) -> torch.Tensor:

    pred_translation, pred_quaternion = split_prediction(prediction)
    target_translation, target_quaternion = split_pose(target)

    pred_frame_euler = quaternion_to_euler(pred_quaternion)
    target_frame_euler = quaternion_to_euler(target_quaternion)

    pred_translation, pred_quaternion = split_prediction(prediction)
    pred_frame_euler = quaternion_to_euler(pred_quaternion)
    pred_frame_transform = prediction_to_transform(prediction)

    target_absolute_transform = pose_to_transform(target)
    target_frame_transform = (
        invert_transform(target_absolute_transform[:, :-1]) @ target_absolute_transform[:, 1:]
    )
    target_frame_translation, target_frame_euler = transform_to_components(target_frame_transform)

    lframe_xyz = rmse(pred_translation, target_frame_translation)
    lframe_euler = rmse(pred_frame_euler, target_frame_euler)
    lframe = (
        self.translation_weight * lframe_xyz
        + self.rotation_weight * lframe_euler
    )

    pred_sequence_transform = compose_sequence_transforms(pred_frame_transform)
    target_origin_inv = invert_transform(target_absolute_transform[:, :1])
    target_sequence_transform = target_origin_inv @ target_absolute_transform[:, 1:]

    pred_sequence_translation, pred_sequence_euler = transform_to_components(pred_sequence_transform)
    target_sequence_translation, target_sequence_euler = transform_to_components(target_sequence_transform)

    lsequence_xyz = rmse(pred_sequence_translation, target_sequence_translation)
    lsequence_euler = rmse(pred_sequence_euler, target_sequence_euler)
    lsequence = (
        self.translation_weight * lsequence_xyz
        + self.rotation_weight * lsequence_euler
    )

    loss = self.frame_weight * lframe + self.sequence_weight * lsequence

    metrics = {
        "loss": loss.detach(),
        "lframe": lframe.detach(),
        "lframe_xyz": lframe_xyz.detach(),
        "lframe_euler": lframe_euler.detach(),
        "lsequence": lsequence.detach(),
        "lsequence_xyz": lsequence_xyz.detach(),
        "lsequence_euler": lsequence_euler.detach(),
    }
    return loss, metrics


## Trainning Loop Implementation

In [ ]:
import torch
import gc
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 0.5
LR_DECAY_STEP = 20
LR_DECAY_GAMMA = 0.5

ENCODER_DIM = 512
EMBED_DIM = 1024
NUM_HEADS = 4
ATTN_BLOCKS = 4
DROPOUT = 0.1
OUTPUT_DIM = 6

# Exclude at least one of the sequences so it can be used as the Test sequence
DATASET_FILES = [
    "MH_01_easy_dataset.pt",
    "MH_02_easy_dataset.pt",
    "MH_03_medium_dataset.pt",
    "MH_04_difficult_dataset.pt",
    "MH_05_difficult_dataset.pt",
    "V1_01_easy_dataset.pt",
    # "V1_02_medium_dataset.pt",
    "V1_03_difficult_dataset.pt",
    "V2_01_easy_dataset.pt",
    "V2_02_medium_dataset.pt",
    "V2_03_difficult_dataset.pt",
]

DATASET_DIR = Path("/content")
CHECKPOINT_DIR = Path("/content/checkpoints")

def main():

  model = CausalFusionModel(
      imu_steps=10,
      img_channels=2,
      attn_blocks=ATTN_BLOCKS,
      embed_dim=EMBED_DIM,
      num_heads=NUM_HEADS,
      context_length=BATCH_SIZE,
      output_dim=OUTPUT_DIM,
      dropout=DROPOUT
  ).to(DEVICE)

  loss_fn = PoseSequenceLoss().to(DEVICE)
  optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
  scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_STEP, gamma=LR_DECAY_GAMMA)

  batcher = EurocMav_Batcher(dataset_files=DATASET_FILES, batch_size=BATCH_SIZE)
  best_val_loss = float("inf")

  print("Starting Trainning")

  for epoch in range(EPOCHS):

    batcher.reset()

    train_metrics = train_one_epoch(model, loss_fn, optimizer, batcher)
    val_loss = validate(model, loss_fn, batcher)

    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"train: {train_metrics}")
    print(f"val: {val_loss}")
    print(f"lr: {optimizer.param_groups[0]['lr']:.8f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            CHECKPOINT_DIR / "best.pt",
        )

    save_checkpoint(
        model,
        optimizer,
        epoch,
        best_val_loss,
        CHECKPOINT_DIR / "latest.pt",
    )

    scheduler.step()

def train_one_epoch(model, loss_fn, optimizer, batcher):
  model.train()

  total_loss = 0.0
  num_steps = 0
  epoch_metrics = {}

  while True:

    batch = batcher.get_batch()

    if batch is None:
      break

    img = batch["img_batch"].to(DEVICE)
    imu = batch["imu_batch"].to(DEVICE)
    target = batch["target_batch"].to(DEVICE)

    optimizer.zero_grad()

    pred = model(img, imu)
    target = target.unsqueeze(0)

    loss, metrics = loss_fn(pred, target)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()

    total_loss += loss.item()

    # Accumulate metrics for averaging
    for k, v in metrics.items():
        if k not in epoch_metrics:
            epoch_metrics[k] = 0.0
        epoch_metrics[k] += v.item()

    num_steps += 1

    del batch, img, imu, target, pred, loss
    gc.collect()

    if torch.cuda.is_available():
      torch.cuda.empty_cache()

  # Average all the metrics over the total number of steps
  if num_steps > 0:
      for k in epoch_metrics:
          epoch_metrics[k] /= num_steps

  return epoch_metrics

def save_checkpoint(model, optimizer, epoch, best_val_loss, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "best_val_loss": best_val_loss,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        },
        path,
    )

def validate(model, loss_fn, batcher):

  val_loss = []

  with torch.no_grad():
    valid_batch = batcher.get_valid_batch()

    img = valid_batch["img_batch"]
    imu = valid_batch["imu_batch"]
    target = valid_batch["target_batch"]

    num_seq = len(batcher.dataset_files)

    img = img.reshape(num_seq, BATCH_SIZE, *img.shape[1:])
    imu = imu.reshape(num_seq, BATCH_SIZE, *imu.shape[1:])
    target = target.reshape(num_seq, BATCH_SIZE + 1, *target.shape[1:])

    target = target.to(DEVICE)
    img = img.to(DEVICE)
    imu = imu.to(DEVICE)

    for i in range(num_seq):
      img_seq = img[i]
      imu_seq = imu[i]
      target_seq = target[i].unsqueeze(0)

      pred = model(img_seq, imu_seq)

      loss, metrics = loss_fn(pred, target_seq)
      val_loss.append(loss.item())

  val_loss = sum(val_loss) / len(val_loss) if len(val_loss) > 0 else 0
  return val_loss


In [ ]:
main()

Starting Trainning
Epoch 1/100
train: {'loss': 1.0787562390605172, 'lframe': 0.8456539907953241, 'lframe_xyz': 0.5708426911752302, 'lframe_euler': 1.1204652861579434, 'lsequence': 3.176676623113863, 'lsequence_xyz': 4.758336703200917, 'lsequence_euler': 1.5950165397518283}
val: 0.9759868144989013
lr: 0.00001000
Epoch 2/100
train: {'loss': 0.9859062086094866, 'lframe': 0.8035400018587218, 'lframe_xyz': 0.4230785952819573, 'lframe_euler': 1.1840014051605057, 'lsequence': 2.627202195780618, 'lsequence_xyz': 3.6619668904241625, 'lsequence_euler': 1.5924374985170888}
val: 0.777236545085907
lr: 0.00001000
Epoch 3/100
train: {'loss': 0.796328196813772, 'lframe': 0.6067303941472546, 'lframe_xyz': 0.3706849368063958, 'lframe_euler': 0.8427758473943878, 'lsequence': 2.5027085490279144, 'lsequence_xyz': 3.418806363593091, 'lsequence_euler': 1.5866107397027067}
val: 0.7105811893939972
lr: 0.00001000
Epoch 4/100
train: {'loss': 0.6681968664729988, 'lframe': 0.4769189002422186, 'lframe_xyz': 0.34299